# Lab 4 — NumPy: Arrays and Vectors
**ARTI 303 — Programming for AI**

**Name:** _[your name]_  **Student ID:** _[your ID]_  **Section:** _[your section]_

---

### Why this lab exists

Python lists are flexible and slow. That's a fine trade when you have
twenty items. It stops being fine at twenty million.

Almost every AI and data library you will ever touch is built on
**NumPy** arrays underneath. Learning NumPy isn't learning one more
library — it's learning the shape of everything that comes after it.

More than speed, though, this lab is about a different way of writing
code. You'll stop writing "for each item, do something" and start
writing "do this to the whole array." That shift is the point.

---

## Setup — do this before running anything

### Your folder should look like this

Everything for this course lives in **one** folder, with **one**
environment shared by every lab:

```
arti303_lab/
├── .venv/              <- one environment for the whole course
├── requirements.txt    <- one recipe, updated as labs add packages
├── lab01.ipynb
├── lab02.ipynb
├── lab03.ipynb
├── lab04.ipynb         <- this lab
├── src/
├── tests/
├── data/
└── logs/
```

Not one environment per lab. One environment, reused every week, with
packages added to it as you go.

### Step 1 — Open the course folder in VS Code

`File → Open Folder` → select **`arti303_lab`** — the folder itself,
not a lab file inside it.

This matters: VS Code only finds `.venv` when it sits at the root of
the folder you have open.

### Step 2 — Activate the environment

Open the terminal (`` Ctrl+` ``) and run:

```bash
source .venv/bin/activate          # macOS / Linux / Git Bash
.venv\Scripts\Activate.ps1         # Windows PowerShell
```

**Now look at your terminal prompt.** It must start with `(.venv)`:

```
(.venv) yourname@computer arti303_lab %
```

If it says `(base)` instead, you are in Anaconda's environment, **not**
your project's. Anything you install will go to the wrong place. Run the
activate command again and check the prompt before continuing.

> Don't have a `.venv` yet? Create it once, from this folder:
> `python -m venv .venv` — then activate it as above.

### Step 3 — Install NumPy and record it

Only once `(.venv)` shows in your prompt:

```bash
python -m pip install numpy
python -m pip freeze > requirements.txt
```

The second command matters as much as the first. An installed package
that isn't in `requirements.txt` doesn't exist as far as anyone else's
machine is concerned.

You are **updating** the existing `requirements.txt`, not making a new
one. It should stay short — a handful of lines. If it comes out with
hundreds of lines, you froze Anaconda's base environment instead of
yours; delete it, fix Step 2, and freeze again.

### Step 4 — Point this notebook at the same environment

The notebook has its own interpreter setting, separate from the
terminal. Set it:

1. `Cmd+Shift+P` (macOS) or `Ctrl+Shift+P` (Windows/Linux)
2. Type **Python: Select Interpreter**
3. Choose the entry showing **`.venv`**

If `.venv` isn't in the list, choose **"Enter interpreter path..."** and
type the full path, for example:

```
/Users/yourname/Desktop/arti303_lab/.venv/bin/python
```

Then restart the kernel and run the check below.

In [1]:
import numpy as np

print("NumPy version:", np.__version__)

NumPy version: 2.4.6


`import numpy as np` — `np` is the universal convention. Every NumPy
example you find online will use it, so use it too.

### Confirm it's recorded

### Check your setup

Run this before anything else. It reports what's actually true on your
machine rather than assuming.

In [2]:
import os
import sys

problems = []

# 1. Is this notebook running inside a virtual environment?
if sys.prefix != sys.base_prefix:
    print("[ok]   Notebook is running inside a virtual environment")
else:
    print("[!]    Notebook is NOT in a virtual environment")
    problems.append("Select the .venv interpreter (Setup, Step 4), then restart the kernel")

print("       using:", sys.executable)
print()

# 2. Does requirements.txt exist, next to this notebook?
if not os.path.exists("requirements.txt"):
    print("[!]    No requirements.txt in this folder")
    problems.append("Run: python -m pip freeze > requirements.txt")
else:
    lines = [l for l in open("requirements.txt").read().splitlines() if l.strip()]
    print(f"[ok]   requirements.txt found ({len(lines)} packages)")

    if not any("numpy" in l.lower() for l in lines):
        print("[!]    numpy is not listed in it")
        problems.append("Install numpy, then freeze again")

    if len(lines) > 100:
        print("[!]    That is a lot of packages - this looks like Anaconda's base")
        problems.append("Delete requirements.txt, activate .venv, and freeze again")

print()
if problems:
    print("Fix these before continuing:")
    for item in problems:
        print("  -", item)
else:
    print("Setup looks good.")

[ok]   Notebook is running inside a virtual environment
       using: /Users/hindalzahrani/desktop/1st Sem/ARTI 303 - Programming for AI/When I run all labs/.venv/bin/python

[ok]   requirements.txt found (1 packages)

Setup looks good.


---
# Part 1 · Why arrays instead of lists

Let's measure the difference rather than assert it.

In [3]:
import time

size = 1_000_000
py_list = list(range(size))
np_array = np.arange(size)

# Doubling every value with a Python loop
start = time.time()
doubled_list = [x * 2 for x in py_list]
list_time = time.time() - start

# The same thing with NumPy
start = time.time()
doubled_array = np_array * 2
array_time = time.time() - start

print(f"Python list: {list_time:.4f} seconds")
print(f"NumPy array: {array_time:.4f} seconds")
print(f"NumPy is about {list_time / array_time:.0f}x faster here")

Python list: 0.0314 seconds
NumPy array: 0.0013 seconds
NumPy is about 24x faster here


Same result, very different cost. Two reasons:

1. **One type, packed together.** A list can hold anything, so Python
   stores a pointer to each item scattered in memory. A NumPy array
   holds one type in one contiguous block.
2. **The loop runs in C, not Python.** `np_array * 2` hands the whole
   operation to compiled code instead of stepping through a Python loop
   a million times.

Notice what the NumPy version looks like:

```python
doubled_array = np_array * 2
```

No loop at all. That's **vectorization** — describing the operation on the whole array and letting NumPy handle the iteration.

---
# Part 2 · Creating arrays

From a list you already have:

In [4]:
list_of_gpas = [4.60, 3.90, 4.85, 4.40, 2.95]
gpas = np.array(list_of_gpas)

print(gpas)
print("type: ", type(gpas))
print("dtype:", gpas.dtype)      # what's inside
print("shape:", gpas.shape)      # dimensions
print("size: ", gpas.size)       # total elements

[4.6  3.9  4.85 4.4  2.95]
type:  <class 'numpy.ndarray'>
dtype: float64
shape: (5,)
size:  5


Three attributes you'll check constantly:

| Attribute | Means |
|---|---|
| `dtype` | the type of every element (`float64`, `int64`, `bool`, ...) |
| `shape` | a tuple of dimensions — `(5,)` is 5 elements in one dimension |
| `size` | total number of elements |

`shape` is the one you'll check most. Most NumPy errors are shape
errors.

### Built-in creators

In [5]:
print("zeros:   ", np.zeros(5))
print("ones:    ", np.ones(3))
print("full:    ", np.full(4, 7.5))
print("arange:  ", np.arange(0, 10, 2))       # start, stop, step
print("linspace:", np.linspace(0, 1, 5))      # start, stop, how many

zeros:    [0. 0. 0. 0. 0.]
ones:     [1. 1. 1.]
full:     [7.5 7.5 7.5 7.5]
arange:   [0 2 4 6 8]
linspace: [0.   0.25 0.5  0.75 1.  ]


`arange` takes a **step**; `linspace` takes a **count** and spaces the
values evenly. When you need exactly N points across a range, use
`linspace` — it hits both endpoints without floating-point drift.

### Two dimensions

A 2D array is a table: rows and columns.

In [6]:
# 4 students, 3 exams each
scores = np.array([
    [88, 92, 79],
    [75, 68, 82],
    [95, 91, 98],
    [60, 72, 65],
])

print(scores)
print()
print("shape:", scores.shape, "-> 4 rows, 3 columns")
print("ndim: ", scores.ndim, "dimensions")

[[88 92 79]
 [75 68 82]
 [95 91 98]
 [60 72 65]]

shape: (4, 3) -> 4 rows, 3 columns
ndim:  2 dimensions


`shape` reads outermost-first: `(4, 3)` means 4 rows, each with 3
values.

### Exercise 2.1
Create:
- `temps` — a 1D array from the list `[22.5, 24.1, 26.8, 25.3]`
- `zeros_grid` — a 2D array of zeros with 3 rows and 5 columns
- `steps` — 6 evenly spaced values from 0 to 100 inclusive

For `zeros_grid`, `np.zeros()` takes a shape tuple: `np.zeros((rows, cols))`.

In [7]:
# TODO: replace None
temps = np.array([22.5, 24.1, 26.8, 25.3])
zeros_grid = np.zeros((3, 5))
steps = np.linspace(0, 100, 6)

print(temps)
print(zeros_grid)
print(steps)

[22.5 24.1 26.8 25.3]
[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]
[  0.  20.  40.  60.  80. 100.]


In [8]:
assert temps is not None and zeros_grid is not None and steps is not None, \
    "fill in every TODO above before running this check"
assert temps.shape == (4,), f"temps shape should be (4,), got {temps.shape}"
assert zeros_grid.shape == (3, 5), f"zeros_grid shape should be (3, 5), got {zeros_grid.shape}"
assert zeros_grid.sum() == 0, "zeros_grid should be all zeros"
assert steps.shape == (6,), f"steps should have 6 values, got {steps.shape}"
assert steps[0] == 0 and steps[-1] == 100, "steps should run from 0 to 100 inclusive"
print("Exercise 2.1 correct.")

Exercise 2.1 correct.


---
# Part 3 · Indexing and slicing

For 1D arrays this works exactly like lists.

In [9]:
gpas = np.array([4.60, 3.90, 4.85, 4.40, 2.95])

print("first:  ", gpas[0])
print("last:   ", gpas[-1])
print("1 to 3: ", gpas[1:4])
print("every 2:", gpas[::2])

first:   4.6
last:    2.95
1 to 3:  [3.9  4.85 4.4 ]
every 2: [4.6  4.85 2.95]


For 2D, you give **two** positions: `array[row, column]`.

In [10]:
scores = np.array([
    [88, 92, 79],
    [75, 68, 82],
    [95, 91, 98],
    [60, 72, 65],
])

print("row 0, col 1: ", scores[0, 1])
print()
print("whole row 2:  ", scores[2])
print("whole col 1:  ", scores[:, 1])      # : means "all rows"
print()
print("first 2 rows, last 2 cols:")
print(scores[:2, 1:])

row 0, col 1:  92

whole row 2:   [95 91 98]
whole col 1:   [92 68 91 72]

first 2 rows, last 2 cols:
[[92 79]
 [68 82]]


`scores[:, 1]` — the colon means "everything in this dimension." That's
how you pull a whole column, and there's no clean equivalent with plain
Python lists.

### Exercise 3.1
From `scores` above, extract:
- `student_2` — all scores for the student at index 2
- `exam_0` — every student's score on exam 0
- `corner` — the last 2 rows and the first 2 columns

In [11]:
# TODO: replace None
student_2 = scores[2]
exam_0 = scores[:, 0]
corner = scores[2:, :2]

print(student_2)
print(exam_0)
print(corner)

[95 91 98]
[88 75 95 60]
[[95 91]
 [60 72]]


In [12]:
assert student_2 is not None and exam_0 is not None and corner is not None, \
    "fill in every TODO above before running this check"
assert list(student_2) == [95, 91, 98], f"got {student_2}"
assert list(exam_0) == [88, 75, 95, 60], f"got {exam_0}"
assert corner.shape == (2, 2), f"corner shape should be (2, 2), got {corner.shape}"
assert corner.tolist() == [[95, 91], [60, 72]], f"got {corner.tolist()}"
print("Exercise 3.1 correct.")

Exercise 3.1 correct.


---
# Part 4 · Views and copies — read this one carefully

This is the part of NumPy that causes silent, hard-to-find bugs. Not
crashes — **wrong numbers, with no error message.**

Slicing a Python list gives you a new list. Slicing a NumPy array gives
you a **view**: a window onto the *same* memory.

In [13]:
# Python list: slicing copies
py = [1, 2, 3, 4, 5]
py_slice = py[1:4]
py_slice[0] = 999
print("list original after editing slice:", py)

list original after editing slice: [1, 2, 3, 4, 5]


In [14]:
# NumPy: slicing does NOT copy
arr = np.array([1, 2, 3, 4, 5])
arr_slice = arr[1:4]
arr_slice[0] = 999
print("array original after editing slice:", arr)

array original after editing slice: [  1 999   3   4   5]


The original array changed. Nothing warned you.

This is deliberate — not copying is what makes NumPy fast on large
data. But it means a function that "just looks at a slice" can quietly
modify your dataset.

### Making a real copy

When you want independence, say so with `.copy()`.

In [15]:
arr = np.array([1, 2, 3, 4, 5])
safe = arr[1:4].copy()
safe[0] = 999

print("original:", arr)
print("copy:    ", safe)

original: [1 2 3 4 5]
copy:     [999   3   4]


Same trap with plain assignment — `b = a` does not copy anything, it
just gives the same array a second name.

In [16]:
a = np.array([1, 2, 3])
b = a              # NOT a copy
c = a.copy()       # a real copy

b[0] = 100
c[1] = 200

print("a:", a)     # changed by b
print("b:", b)
print("c:", c)     # independent

a: [100   2   3]
b: [100   2   3]
c: [  1 200   3]


### Exercise 4.1
`readings` holds sensor data. You need the first three values
**cleaned** (set to 0.0) for a separate calculation — but `readings`
itself must stay untouched.

Create `first_three_clean` as an independent copy, zero it out, and
leave `readings` unchanged.

In [17]:
readings = np.array([21.0, 100.0, 23.5, 22.0, 19.5])

# TODO: make an independent copy of the first three values
first_three_clean = readings[:3].copy()

# TODO: set every value in first_three_clean to 0.0
first_three_clean[:] = 0.0


print("first_three_clean:", first_three_clean)
print("readings:         ", readings)

first_three_clean: [0. 0. 0.]
readings:          [ 21.  100.   23.5  22.   19.5]


In [18]:
assert first_three_clean is not None, "first_three_clean is still None"
assert list(first_three_clean) == [0.0, 0.0, 0.0], f"got {first_three_clean}"
assert list(readings) == [21.0, 100.0, 23.5, 22.0, 19.5], \
    f"readings should be unchanged - you modified the original: {readings}"
print("Exercise 4.1 correct.")

Exercise 4.1 correct.


> **Rule of thumb:** if a function receives an array and modifies it,
> assume the caller's data changed too, unless you copied first. When a
> value is mysteriously wrong later in a pipeline, this is the first
> thing to check.

---
# Part 5 · Vectorized operations

Arithmetic applies to every element at once.

In [19]:
gpas = np.array([4.60, 3.90, 4.85, 4.40, 2.95])

print("as percent:  ", gpas / 5.0 * 100)
print("plus 0.1:    ", gpas + 0.1)
print("squared:     ", gpas ** 2)
print("rounded:     ", np.round(gpas))

as percent:   [92. 78. 97. 88. 59.]
plus 0.1:     [4.7  4.   4.95 4.5  3.05]
squared:      [21.16   15.21   23.5225 19.36    8.7025]
rounded:      [5. 4. 5. 4. 3.]


Arrays of the same shape combine element by element.

In [20]:
midterm = np.array([80, 65, 92, 55])
final   = np.array([88, 70, 95, 60])

print("difference:", final - midterm)
print("average:   ", (midterm + final) / 2)

difference: [8 5 3 5]
average:    [84.  67.5 93.5 57.5]


### Useful mathematical functions

In [21]:
values = np.array([1.0, 4.0, 9.0, 16.0])

print("sqrt:", np.sqrt(values))
print("log: ", np.log(values))
print("exp: ", np.exp(np.array([0.0, 1.0, 2.0])))
print("abs: ", np.abs(np.array([-3.0, 2.0, -1.0])))

sqrt: [1. 2. 3. 4.]
log:  [0.         1.38629436 2.19722458 2.77258872]
exp:  [1.         2.71828183 7.3890561 ]
abs:  [3. 2. 1.]


### Exercise 5.1
`raw_scores` are out of 100. Convert them to our 5.00 GPA scale and
round to 2 decimal places, storing the result in `gpa_scale`.

A score of 100 maps to 5.00, a score of 0 maps to 0.00.

Do it **without a loop** — one expression.

In [22]:
raw_scores = np.array([88, 65, 97, 55, 72])

# TODO: convert to the 5.00 scale, rounded to 2 decimals
gpa_scale = np.round(raw_scores / 100 * 5, 2)


print(gpa_scale)

[4.4  3.25 4.85 2.75 3.6 ]


In [23]:
assert gpa_scale is not None, "gpa_scale is still None"
expected = [4.40, 3.25, 4.85, 2.75, 3.60]
assert np.allclose(gpa_scale, expected), f"expected {expected}, got {list(gpa_scale)}"
print("Exercise 5.1 correct.")

Exercise 5.1 correct.


> `np.allclose` compares floating-point arrays with a small tolerance.
> Use it instead of `==` for floats — `0.1 + 0.2 == 0.3` is `False` in
> every language that uses binary floating point, Python included.

---
# Part 6 · Boolean masks — filtering without loops

Comparing an array to a value gives you an array of `True`/`False`.

In [24]:
gpas = np.array([4.60, 3.90, 4.85, 4.40, 2.95])

mask = gpas >= 4.50
print("mask:", mask)

mask: [ True False  True False False]


That mask can then select elements.

In [25]:
print("Dean's List GPAs:", gpas[gpas >= 4.50])
print("how many:        ", np.sum(gpas >= 4.50))
print("their positions: ", np.where(gpas >= 4.50)[0])

Dean's List GPAs: [4.6  4.85]
how many:         2
their positions:  [0 2]


`np.sum()` on a boolean array counts the `True` values, because `True`
is 1 and `False` is 0. Counting a condition is that short.

### Combining conditions

Use `&` (and), `|` (or), `~` (not) — **not** the words `and`/`or`. And
each condition needs its own brackets.

In [26]:
gpas = np.array([4.60, 3.90, 4.85, 4.40, 2.95])

print("between 3.5 and 4.5:", gpas[(gpas >= 3.50) & (gpas < 4.50)])
print("below 3.5 or above 4.8:", gpas[(gpas < 3.50) | (gpas > 4.80)])
print("NOT Dean's List:", gpas[~(gpas >= 4.50)])

between 3.5 and 4.5: [3.9 4.4]
below 3.5 or above 4.8: [4.85 2.95]
NOT Dean's List: [3.9  4.4  2.95]


### Replacing values conditionally

`np.where(condition, value_if_true, value_if_false)`.

In [27]:
readings = np.array([21.0, 100.0, 23.5, 22.0, 19.5])

# Treat anything above 50 as a sensor fault, replace with the mean of the rest
valid_mean = readings[readings <= 50].mean()
cleaned = np.where(readings > 50, valid_mean, readings)

print("original:", readings)
print("cleaned: ", np.round(cleaned, 2))

original: [ 21.  100.   23.5  22.   19.5]
cleaned:  [21.  21.5 23.5 22.  19.5]


### Exercise 6.1
Using `class_gpas`:
- `dean_list` — the GPAs at or above 4.50
- `at_risk_count` — how many are **below 2.00**
- `capped` — every GPA above 5.00 replaced with 5.00 (data-entry errors), others unchanged

All three without a loop.

In [28]:
class_gpas = np.array([4.60, 1.85, 4.85, 5.40, 2.95, 4.50, 1.20])

# TODO: replace None
dean_list = class_gpas[class_gpas >= 4.50]
at_risk_count = np.sum(class_gpas < 2.00)
capped = np.where(class_gpas > 5.00, 5.00, class_gpas)

print("dean_list:    ", dean_list)
print("at_risk_count:", at_risk_count)
print("capped:       ", capped)

dean_list:     [4.6  4.85 5.4  4.5 ]
at_risk_count: 2
capped:        [4.6  1.85 4.85 5.   2.95 4.5  1.2 ]


In [29]:
assert dean_list is not None and at_risk_count is not None and capped is not None, \
    "fill in every TODO above before running this check"
assert np.allclose(dean_list, [4.60, 4.85, 5.40, 4.50]), f"got {list(dean_list)}"
assert at_risk_count == 2, f"expected 2 at risk, got {at_risk_count}"
assert np.allclose(capped, [4.60, 1.85, 4.85, 5.00, 2.95, 4.50, 1.20]), f"got {list(capped)}"
print("Exercise 6.1 correct.")

Exercise 6.1 correct.


---
# Part 7 · Aggregation and `axis`

Summarising a 1D array is straightforward.

In [30]:
gpas = np.array([4.60, 3.90, 4.85, 4.40, 2.95])

print("sum:   ", gpas.sum())
print("mean:  ", gpas.mean().round(3))
print("min:   ", gpas.min())
print("max:   ", gpas.max())
print("std:   ", gpas.std().round(3))
print("argmax:", gpas.argmax(), "<- position of the highest")

sum:    20.7
mean:   4.14
min:    2.95
max:    4.85
std:    0.672
argmax: 2 <- position of the highest


`argmax` gives the **position**, not the value — useful when you need
to look up who it belongs to.

### `axis` on 2D arrays

This is the concept students most often get backwards, so it's worth
slowing down.

- `axis=0` collapses **rows** → one result per **column**
- `axis=1` collapses **columns** → one result per **row**

Think of it as "the axis that disappears."

`scores` has shape `(4, 3)` — 4 students, 3 exams.

In [31]:
scores = np.array([
    [88, 92, 79],
    [75, 68, 82],
    [95, 91, 98],
    [60, 72, 65],
])

print("overall mean:     ", scores.mean().round(2))
print()
print("axis=0 ->", scores.mean(axis=0).round(2), " shape", scores.mean(axis=0).shape)
print("   one value per EXAM (the 4 students collapsed)")
print()
print("axis=1 ->", scores.mean(axis=1).round(2), " shape", scores.mean(axis=1).shape)
print("   one value per STUDENT (the 3 exams collapsed)")

overall mean:      80.42

axis=0 -> [79.5  80.75 81.  ]  shape (3,)
   one value per EXAM (the 4 students collapsed)

axis=1 -> [86.33 75.   94.67 65.67]  shape (4,)
   one value per STUDENT (the 3 exams collapsed)


Check the shapes: `(4, 3)` with `axis=0` gives `(3,)`, and with
`axis=1` gives `(4,)`. The axis you name is the one that vanishes.

When you're unsure which you want, ask: *how many results do I expect?*
Three exam averages → the answer must have shape `(3,)` → `axis=0`.

### Exercise 7.1
From `scores`:
- `per_student` — each student's average, rounded to 2 decimals
- `per_exam_max` — the highest score on each exam
- `best_student` — the **index** of the student with the highest average
- `hardest_exam` — the **index** of the exam with the lowest average

In [32]:
# TODO: replace None
per_student = scores.mean(axis=1).round(2)
per_exam_max = scores.max(axis=0)
best_student = scores.mean(axis=1).argmax()
hardest_exam = scores.mean(axis=0).argmin()

print("per_student: ", per_student)
print("per_exam_max:", per_exam_max)
print("best_student:", best_student)
print("hardest_exam:", hardest_exam)

per_student:  [86.33 75.   94.67 65.67]
per_exam_max: [95 92 98]
best_student: 2
hardest_exam: 0


In [33]:
assert per_student is not None and per_exam_max is not None \
    and best_student is not None and hardest_exam is not None, \
    "fill in every TODO above before running this check"
assert per_student.shape == (4,), f"per_student should have 4 values, got {per_student.shape}"
assert np.allclose(per_student, [86.33, 75.0, 94.67, 65.67]), f"got {list(per_student)}"
assert list(per_exam_max) == [95, 92, 98], f"got {list(per_exam_max)}"
assert best_student == 2, f"expected student 2, got {best_student}"
assert hardest_exam == 0, f"expected exam 0, got {hardest_exam}"
print("Exercise 7.1 correct.")

Exercise 7.1 correct.


---
# Part 8 · Broadcasting

Arrays of different shapes can often still combine. NumPy stretches the
smaller one to fit — without actually copying the data.

You've used it already: `gpas * 2` broadcasts the single number `2`
across every element.

In [34]:
scores = np.array([
    [88, 92, 79],
    [75, 68, 82],
    [95, 91, 98],
    [60, 72, 65],
])

# One bonus per exam, applied to all students
bonus = np.array([2, 0, 5])          # shape (3,)

print("scores shape:", scores.shape)
print("bonus shape: ", bonus.shape)
print()
print(scores + bonus)

scores shape: (4, 3)
bonus shape:  (3,)

[[ 90  92  84]
 [ 77  68  87]
 [ 97  91 103]
 [ 62  72  70]]


`(4, 3) + (3,)` works: the bonus row is applied to every student.

**The rule:** compare shapes from the right. Dimensions must either
match, or one of them must be 1.

| A | B | Works? |
|---|---|---|
| `(4, 3)` | `(3,)` | yes — applied to each row |
| `(4, 3)` | `(4, 1)` | yes — applied to each column |
| `(4, 3)` | `(4,)` | **no** — 3 vs 4 |

That third case is the one that bites. To apply something per *student*,
reshape it to a column.

In [35]:
per_student_bonus = np.array([1, 2, 0, 3])          # shape (4,)

try:
    scores + per_student_bonus
except ValueError as e:
    print("Error:", e)

print()
column = per_student_bonus.reshape(4, 1)           # shape (4, 1)
print("reshaped to", column.shape)
print(scores + column)

Error: operands could not be broadcast together with shapes (4,3) (4,) 

reshaped to (4, 1)
[[89 93 80]
 [77 70 84]
 [95 91 98]
 [63 75 68]]


### Exercise 8.1
Each exam has a different maximum mark: exam 0 is out of 100, exam 1
out of 100, exam 2 out of 120.

Convert `scores` into **percentages** for each exam, rounded to 1
decimal place. Use broadcasting, not a loop.

In [36]:
max_marks = np.array([100, 100, 120])

# TODO: convert every score to a percentage of its exam's maximum
percentages = np.round(scores / max_marks * 100, 1)

print(percentages)

[[88.  92.  65.8]
 [75.  68.  68.3]
 [95.  91.  81.7]
 [60.  72.  54.2]]


In [37]:
assert percentages is not None, "percentages is still None"
assert percentages.shape == (4, 3), f"shape should be (4, 3), got {percentages.shape}"
assert np.allclose(percentages[0], [88.0, 92.0, 65.8]), f"row 0 wrong: {list(percentages[0])}"
assert np.allclose(percentages[2], [95.0, 91.0, 81.7]), f"row 2 wrong: {list(percentages[2])}"
print("Exercise 8.1 correct.")

Exercise 8.1 correct.


---
# Part 9 · Reshaping

The same data can be viewed in different shapes, as long as the total
number of elements stays the same.

In [38]:
flat = np.arange(12)
print("flat:  ", flat, flat.shape)
print()
print("3 x 4:")
print(flat.reshape(3, 4))
print()
print("4 x 3:")
print(flat.reshape(4, 3))
print()
print("-1 lets NumPy work it out:")
print(flat.reshape(2, -1).shape, "  (2 rows, columns inferred)")

flat:   [ 0  1  2  3  4  5  6  7  8  9 10 11] (12,)

3 x 4:
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]

4 x 3:
[[ 0  1  2]
 [ 3  4  5]
 [ 6  7  8]
 [ 9 10 11]]

-1 lets NumPy work it out:
(2, 6)   (2 rows, columns inferred)


`-1` means "figure this dimension out from the others." Handy when you
know you want 2 rows but don't want to count the columns.

Flattening back:

In [39]:
grid = np.arange(6).reshape(2, 3)
print(grid)
print()
print("ravel: ", grid.ravel())       # a view where possible
print("flatten:", grid.flatten())    # always a copy

[[0 1 2]
 [3 4 5]]

ravel:  [0 1 2 3 4 5]
flatten: [0 1 2 3 4 5]


> `ravel()` returns a view when it can, `flatten()` always copies. If
> you plan to modify the result, `flatten()` is the safe choice — for
> the same reason as Part 4.

### Exercise 9.1
`readings_flat` holds 12 sensor readings taken over 4 days, 3 per day,
in order.

- `by_day` — reshaped to 4 rows (one per day), 3 columns
- `daily_mean` — the mean for each day, rounded to 2 decimals

In [40]:
readings_flat = np.array([
    21.0, 22.5, 23.0,
    19.5, 20.0, 21.5,
    25.0, 26.5, 24.0,
    18.0, 17.5, 19.0,
])

# TODO: replace None
by_day = readings_flat.reshape(4, 3)
daily_mean = by_day.mean(axis=1).round(2)

print(by_day)
print(daily_mean)

[[21.  22.5 23. ]
 [19.5 20.  21.5]
 [25.  26.5 24. ]
 [18.  17.5 19. ]]
[22.17 20.33 25.17 18.17]


In [41]:
assert by_day is not None and daily_mean is not None, \
    "fill in every TODO above before running this check"
assert by_day.shape == (4, 3), f"shape should be (4, 3), got {by_day.shape}"
assert np.allclose(by_day[0], [21.0, 22.5, 23.0]), "first day's readings are wrong"
assert np.allclose(daily_mean, [22.17, 20.33, 25.17, 18.17]), f"got {list(daily_mean)}"
print("Exercise 9.1 correct.")

Exercise 9.1 correct.


---
# Part 10 · Matrix operations

`*` multiplies element by element. `@` does matrix multiplication.
They are not the same thing, and mixing them up is a classic bug.

In [42]:
A_mat = np.array([[1, 2],
                  [3, 4]])
B_mat = np.array([[5, 6],
                  [7, 8]])

print("element-wise (A * B):")
print(A_mat * B_mat)
print()
print("matrix product (A @ B):")
print(A_mat @ B_mat)

element-wise (A * B):
[[ 5 12]
 [21 32]]

matrix product (A @ B):
[[19 22]
 [43 50]]


### Transpose

`.T` flips rows and columns.

In [43]:
scores = np.array([
    [88, 92, 79],
    [75, 68, 82],
    [95, 91, 98],
    [60, 72, 65],
])

print("scores.shape:  ", scores.shape)
print("scores.T.shape:", scores.T.shape)
print()
print(scores.T)

scores.shape:   (4, 3)
scores.T.shape: (3, 4)

[[88 75 95 60]
 [92 68 91 72]
 [79 82 98 65]]


### A weighted total

A practical use of `@`: each exam carries a different weight.

In [44]:
weights = np.array([0.25, 0.25, 0.50])     # exam 2 is worth half

final_marks = scores @ weights
print("weights sum to:", weights.sum())
print("final marks:   ", final_marks.round(2))

weights sum to: 1.0
final marks:    [84.5  76.75 95.5  65.5 ]


`(4, 3) @ (3,)` → `(4,)`: one weighted total per student. The inner
dimensions must match — 3 exams against 3 weights.

### Exercise 10.1
Coursework carries 40% and the final exam 60%.

`assessment` holds `[coursework, final]` for each of 5 students.

- `overall` — each student's weighted total, rounded to 2 decimals
- `passed` — the overall marks of students scoring **60 or above**

Use `@` for the weighting.

In [45]:
assessment = np.array([
    [80, 88],
    [65, 55],
    [92, 95],
    [45, 62],
    [70, 58],
])
weights = np.array([0.40, 0.60])

# TODO: replace None
overall = np.round(assessment @ weights, 2)
passed = overall[overall >= 60]

print("overall:", overall)
print("passed: ", passed)

overall: [84.8 59.  93.8 55.2 62.8]
passed:  [84.8 93.8 62.8]


In [46]:
assert overall is not None and passed is not None, \
    "fill in every TODO above before running this check"
assert overall.shape == (5,), f"overall should have 5 values, got {overall.shape}"
assert np.allclose(overall, [84.8, 59.0, 93.8, 55.2, 62.8]), f"got {list(overall)}"
assert np.allclose(passed, [84.8, 93.8, 62.8]), f"got {list(passed)}"
print("Exercise 10.1 correct.")

Exercise 10.1 correct.


---
# Part 11 · Work it out by hand first

Everything so far, you ran the code and trusted the number that came
out. This part flips that around.

For each task below: **solve it on paper first**, type your answer into
`my_answer`, and only then run the comparison. If they match, you
understood the operation. If they don't, you'll see exactly which
element you got wrong.

Small whole numbers throughout, so the arithmetic is quick.

### Our two matrices

$$A = \begin{bmatrix} 1 & 2 \\ 3 & 4 \end{bmatrix}
\qquad
B = \begin{bmatrix} 5 & 6 \\ 7 & 8 \end{bmatrix}$$

In [47]:
A = np.array([[1, 2],
              [3, 4]])

B = np.array([[5, 6],
              [7, 8]])

print("A =\n", A)
print("B =\n", B)

A =
 [[1 2]
 [3 4]]
B =
 [[5 6]
 [7 8]]


## 11.1 — Addition

Addition is element by element: the value at each position in `A` plus
the value at the same position in `B`.

$$A + B = \begin{bmatrix} 1+5 & 2+6 \\ 3+7 & 4+8 \end{bmatrix}$$

Work out all four numbers, then fill them in below.

In [50]:
# Your hand-calculated answer - replace the zeros
my_answer = np.array([[6, 8],
                      [10, 12]])

print("mine: \n", my_answer)
print("NumPy:\n", A + B)
print("match?", np.array_equal(my_answer, A + B))

mine: 
 [[ 6  8]
 [10 12]]
NumPy:
 [[ 6  8]
 [10 12]]
match? True


In [51]:
assert np.array_equal(my_answer, A + B), \
    "not a match yet - compare your answer with NumPy's, position by position"
print("11.1 correct.")

11.1 correct.


## 11.2 — Subtraction

Same idea: `A - B`, element by element. Watch the signs.

In [53]:
my_answer = np.array([[-4, -4],
                      [-4, -4]])

print("mine: \n", my_answer)
print("NumPy:\n", A - B)
print("match?", np.array_equal(my_answer, A - B))

mine: 
 [[-4 -4]
 [-4 -4]]
NumPy:
 [[-4 -4]
 [-4 -4]]
match? True


In [54]:
assert np.array_equal(my_answer, A - B), \
    "not a match - remember every result here is negative"
print("11.2 correct.")

11.2 correct.


## 11.3 — Scalar multiplication

`2 * A` multiplies **every** element by 2. Nothing clever happens.

In [59]:
my_answer = np.array([[2,4],
                     [6, 8]])

print("mine: \n", my_answer)
print("NumPy:\n", 2 * A)
print("match?", np.array_equal(my_answer, 2 * A))

mine: 
 [[2 4]
 [6 8]]
NumPy:
 [[2 4]
 [6 8]]
match? True


In [60]:
assert np.array_equal(my_answer, 2 * A), "each element should simply double"
print("11.3 correct.")

11.3 correct.


## 11.4 — Element-wise multiplication: `A * B`

This is **not** matrix multiplication. Each element of `A` is multiplied
by the element in the **same position** in `B`.

$$A * B = \begin{bmatrix} 1\times5 & 2\times6 \\ 3\times7 & 4\times8 \end{bmatrix}$$

In [62]:
my_answer = np.array([[5, 12],
                      [21, 32]])

print("mine: \n", my_answer)
print("NumPy:\n", A * B)
print("match?", np.array_equal(my_answer, A * B))

mine: 
 [[ 5 12]
 [21 32]]
NumPy:
 [[ 5 12]
 [21 32]]
match? True


In [63]:
assert np.array_equal(my_answer, A * B), \
    "position by position: A[0,0]*B[0,0], A[0,1]*B[0,1], and so on"
print("11.4 correct.")

11.4 correct.


## 11.5 — Matrix multiplication: `A @ B`

Now the one that's actually different.

Each result element is a **row of A** combined with a **column of B**:
multiply the pairs, then add them up.

$$(A@B)_{0,0} = \text{row 0 of } A \cdot \text{col 0 of } B
= (1\times5) + (2\times7) = 5 + 14 = 19$$

$$(A@B)_{0,1} = (1\times6) + (2\times8) = 6 + 16 = 22$$

Two down. Work out the bottom row yourself:

$$(A@B)_{1,0} = (3\times5) + (4\times7) = ?$$
$$(A@B)_{1,1} = (3\times6) + (4\times8) = ?$$

Write out each element's arithmetic in the comment before filling in
the array — the working is the part that matters here, not the final
four numbers.

In [65]:
# Show your working, one line per element:
# [0,0] = (1x5) + (2x7) = 19
# [0,1] = (1x6) + (2x8) = 22
# [1,0] = ?
# [1,1] = ?

my_answer = np.array([[19, 22],
                      [43, 50]])

print("mine: \n", my_answer)
print("NumPy:\n", A @ B)
print("match?", np.array_equal(my_answer, A @ B))

mine: 
 [[19 22]
 [43 50]]
NumPy:
 [[19 22]
 [43 50]]
match? True


In [66]:
assert np.array_equal(my_answer, A @ B), \
    "each element is row-of-A times column-of-B, summed - check your working above"
print("11.5 correct.")

11.5 correct.


### Compare 11.4 and 11.5

| | result |
|---|---|
| `A * B` | `[[5, 12], [21, 32]]` |
| `A @ B` | `[[19, 22], [43, 50]]` |

Same two matrices, completely different answers — and **neither one
raises an error**. If you use the wrong operator, your program runs
happily and gives you wrong numbers.

This is why the hand calculation matters: it's the only way to know
which one you actually wanted.

## 11.6 — Non-square matrices

Shapes have to line up for `@`. The **inner** dimensions must match, and
the result takes the **outer** ones:

$$(2 \times \mathbf{3}) \;@\; (\mathbf{3} \times 2) \;\rightarrow\; (2 \times 2)$$

$$C = \begin{bmatrix} 1 & 2 & 3 \\ 4 & 5 & 6 \end{bmatrix}
\qquad
D = \begin{bmatrix} 7 & 8 \\ 9 & 10 \\ 11 & 12 \end{bmatrix}$$

Now each element sums **three** products instead of two:

$$(C@D)_{0,0} = (1\times7) + (2\times9) + (3\times11) = 7 + 18 + 33 = 58$$

Work out the other three.

In [67]:
C = np.array([[1, 2, 3],
              [4, 5, 6]])

D = np.array([[7, 8],
              [9, 10],
              [11, 12]])

print("C shape:", C.shape, " D shape:", D.shape)
print("so C @ D will be shape:", (C.shape[0], D.shape[1]))

C shape: (2, 3)  D shape: (3, 2)
so C @ D will be shape: (2, 2)


In [69]:
# [0,0] = (1x7) + (2x9)  + (3x11) = 7 + 18 + 33  = 58
# [0,1] = (1x8) + (2x10) + (3x12) = 8 + 20 + 36  = 64
# [1,0] = (4x7) + (5x9)  + (6x11) = 28 + 45 + 66 = 139
# [1,1] = (4x8) + (5x10) + (6x12) = 32 + 50 + 72 = 154

my_answer = np.array([[58, 64],
                      [139, 154]])

print("mine: \n", my_answer)
print("NumPy:\n", C @ D)
print("match?", np.array_equal(my_answer, C @ D))

mine: 
 [[ 58  64]
 [139 154]]
NumPy:
 [[ 58  64]
 [139 154]]
match? True


In [70]:
assert np.array_equal(my_answer, C @ D), \
    "three products summed per element - check your working"
print("11.6 correct.")

11.6 correct.


## 11.7 — When the shapes don't line up

`D @ C` is `(3x2) @ (2x3)` — inner dimensions are 2 and 2, so it works,
but gives a 3x3 result.

What about `C @ C`? That's `(2x3) @ (2x3)`: inner dimensions 3 and 2.
They don't match.

Predict what happens, then run it.

In [71]:
print("D @ C shape:", (D @ C).shape)
print()

try:
    C @ C
except ValueError as e:
    print("C @ C fails:", e)

D @ C shape: (3, 3)

C @ C fails: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 2 is different from 3)


> Matrix multiplication is **not commutative**: `A @ B` and `B @ A` are
> different, and sometimes only one of them is even valid. Element-wise
> `*` doesn't have this problem, which is part of why mixing the two up
> is so easy.

In [73]:
# Confirm it for yourself: is A @ B the same as B @ A?
# Work out B @ A by hand first.
# [0,0] = (5x1) + (6x3) = 5 + 18 = 23
# [0,1] = (5x2) + (6x4) = 10 + 24 = 34
# [1,0] = (7x1) + (8x3) = 7 + 24 = 31
# [1,1] = (7x2) + (8x4) = 14 + 32 = 46

my_answer = np.array([[23, 34],
                      [31, 46]])

print("B @ A (mine): \n", my_answer)
print("B @ A (NumPy):\n", B @ A)
print("match?", np.array_equal(my_answer, B @ A))
print()
print("Is A @ B equal to B @ A?", np.array_equal(A @ B, B @ A))

B @ A (mine): 
 [[23 34]
 [31 46]]
B @ A (NumPy):
 [[23 34]
 [31 46]]
match? True

Is A @ B equal to B @ A? False


In [74]:
assert np.array_equal(my_answer, B @ A), "check your working for B @ A"
assert not np.array_equal(A @ B, B @ A), "these should NOT be equal"
print("11.7 correct - and you have just shown A @ B is not B @ A.")

11.7 correct - and you have just shown A @ B is not B @ A.


---
# Before you finish

- [ ] Every self-check cell prints "correct"
- [ ] `numpy` is installed **and** recorded in `requirements.txt`
- [ ] **Restart Kernel and Run All** — the whole notebook runs with no errors
- [ ] Committed and pushed